## Steel Plate Analysis Technical Report
Spencer Mundel and Damon Lewis

### Dataset Description
|                | Steel Plate Dataset |
|----------------|---------------------|
| Instances      | 1941                |
| Unique Classes | 7                   |
| Attributes     | 27                  |

The dataset does not contain any missing classes. Most of the data is continous, with only the steel type values (`TypeOfSteel_A300` and `TypeOfSteel_A400`) being binary classes. 

Overall, the KNN classifier peformed the best, with around 70% accuracy versus ~40% for the other classes.

### EDA Development

In [ ]:
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from mysklearn.mypytable import MyPyTable
from mysklearn.myutils import *

In [ ]:
CLASSES = ["Pastry","Z_Scratch","K_Scratch","Stains","Dirtiness","Bumps","Other_Faults"]

data = MyPyTable().load_from_file("plate-data.csv")
data = onehot_to_categorical(data, CLASSES)

#### Distribution of Fault Type and X Minimum

In [ ]:
for attribute in data.column_names:
    if attribute == "Class":
        continue

    sns.boxplot(data=eda_group_data_by_class(data, attribute))
    plt.xticks(rotation=15)
    plt.title(f"{attribute} and Fault Type")
    plt.xlabel("Fault Type")
    plt.ylabel(f"{attribute}")
    plt.show()

#### Count Distributions for All Attributes

In [ ]:
import copy
col_count = 6

tmp_classes = copy.deepcopy(data.column_names)
tmp_classes.remove('Class')
tmp_fig, ax = plt.subplots((len(tmp_classes)//col_count)+1, col_count, layout='constrained', figsize=(8,8))
tmp_fig.set_size_inches(11, 8.5)
for col in range(len(tmp_classes)):
    tmp = ax.flat[col]
    tmp.hist(data.get_column(data.column_names[col]))
    tmp.set_title(data.column_names[col])
plt.savefig('dist.png')

#### Occurrences for Each Class

In [ ]:
counts = {k: 0 for k in CLASSES}
for result in data.get_column('Class'):
    counts[result]+=1

fig, ax = plt.subplots()
ax.set_title("Class Counts")
ax.bar(counts.keys(), counts.values())
ax.tick_params('x', rotation=20)
fig.savefig('counts.png')

### Model Testing and Results

In [ ]:
from copy import deepcopy
from mysklearn import myutils, myclassifiers, mypytable

CLASSES = ["Pastry","Z_Scratch","K_Scratch","Stains","Dirtiness","Bumps","Other_Faults"]
CLASS_HEADER = "Class"

data = mypytable.MyPyTable().load_from_file("plate-data.csv")
data = myutils.onehot_to_categorical(data, CLASSES)
features = deepcopy(data.column_names)
features.pop(features.index(CLASS_HEADER))

In [ ]:
binned = mypytable.MyPyTable(data.column_names, data.data)

for feature in features:
    if feature == "TypeOfSteel_A300" or feature == "TypeOfSteel_A400":
        continue

    col_values = data.get_column(feature)
    binned_col = myutils.equal_width_bin(col_values, 10)
    binned.replace_column(feature, binned_col)

In [ ]:
normalized = mypytable.MyPyTable(data.column_names, data.data)

for feature in features:
    if feature == "TypeOfSteel_A300" or feature == "TypeOfSteel_A400":
        continue

    col_values = data.get_column(feature)
    normalized_col = myutils.normalize_list(col_values)
    normalized.replace_column(feature, normalized_col)

#### Decision Tree

In [ ]:
myutils.stratified_kfold_tester(myclassifiers.MyDecisionTreeClassifier(len(features)),
                                binned,
                                CLASS_HEADER,
                                CLASSES,
                                10)

### KNN

In [ ]:
myutils.stratified_kfold_tester(myclassifiers.MyKNeighborsClassifier(n_neighbors=3),
                                normalized,
                                CLASS_HEADER,
                                CLASSES,
                                10)

#### Random Forest

In [ ]:
myutils.stratified_kfold_tester(myclassifiers.MyRandomForestClassifier(10, 5, 5),
                                binned,
                                CLASS_HEADER,
                                CLASSES,
                                10)

#### Clustered Random Forest

In [ ]:
myutils.stratified_kfold_tester(myclassifiers.MyClusteredRandomForestClassifier(10, 5, 5, 2),
                                data,
                                CLASS_HEADER,
                                CLASSES,
                                10)